In [0]:
from pyspark.sql import functions as f

In [0]:
supplychain_table = spark.table("supplychain_table")
dim_products = supplychain_table.select('Product_Card_Id',
    'Product_Category_Id',
    'Product_Description',
    'Product_Image',
    "Category_Name",
    'Product_Name',
    'Product_Price',
    'Product_Status').dropDuplicates(['Product_Card_Id']).withColumn('Product_key', f.monotonically_increasing_id())

In [0]:
dim_products.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("dim_products")


In [0]:
supplychain_table = spark.table("supplychain_table")
dim_customers = supplychain_table.select('Customer_Id',
'Customer_City',
 'Customer_Country',
 'Customer_Email',
 'Customer_Fname',
 'Customer_Lname',
 'Customer_Segment',
 'Customer_State').dropDuplicates(['Customer_Id']).withColumn('customer_key', f.monotonically_increasing_id())

dim_customers.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("dim_customers")


In [0]:
dim_locations = supplychain_table.select(
    "Market",
    "Order_Region",
    "Order_Country",
    "Customer_State",
    "Customer_City",
    "Latitude",
    "Longitude"
).dropDuplicates(["Market", "Order_Region", "Order_Country", "Customer_State", "Customer_City", "Latitude", "longitude"]).withColumn("location_key", f.monotonically_increasing_id()) ##because the geographic locations dont have a unique key so i had to look through ll the columns to find a unique key

dim_locations.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("dim_locations")

In [0]:
fact_orders = supplychain_table.join(dim_customers, "Customer_Id", "left") \
    .join(dim_products, "Product_Card_Id", "left") \
    .join(dim_locations, ["Market", "Order_Region", "Order_Country", "Customer_State", "Customer_City", "Latitude", "longitude"], "left") \
        .select(
        "Order_Id",
        "Order_Item_Id",
        "Product_key",
        "customer_key",
        "location_key",
        "order_date_DateOrders",
        "shipping_date_DateOrders",
        "Order_Item_Quantity",
        "Sales_per_customer",
        "Sales",
        "Order_Item_Total",
        "Benefit_per_order"
    )
        
fact_orders.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("fact_orders")

In [0]:
fact_shipments = supplychain_table.select(
    "Order_Item_Id",
    "Order_Id",
    "Delivery_Status",
    "Days_for_shipping_real",
    "Days_for_shipment_scheduled",
    "Late_delivery_risk"
)

fact_shipments.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("fact_shipments")

In [0]:
# Check for orphan customer keys in fact_orders
orphans = fact_orders.filter("customer_key IS NULL").count()
print(f"Unmapped customer rows: {orphans}")

In [0]:
%sql
--joining all together for future analysis 

CREATE OR REPLACE VIEW v_supply_chain_performance AS
SELECT 
    f1.Order_Id,
    f1.Order_Item_Id,
    f1.order_date_DateOrders,
    p.Product_Name,
    p.Category_Name,
    p.product_category_id,    
    c.Customer_Segment,
    l.Market,
    l.Order_Region,
    f1.Order_Item_Total,
    f1.Benefit_per_order,
    f2.Delivery_Status,
    f2.Days_for_shipping_real,
    f2.Days_for_shipment_scheduled,
    f2.Late_delivery_risk
FROM fact_orders f1
JOIN fact_shipments f2 ON f1.Order_Item_Id = f2.Order_Item_Id
LEFT JOIN dim_products p ON f1.product_key = p.product_key
LEFT JOIN dim_customers c ON f1.customer_key = c.customer_key
LEFT JOIN dim_locations l ON f1.location_key = l.location_key;